# TF-IDF とコサイン類似度で似た文書を探す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_search/tfidf_similarity.ipynb)

NLP クックブック 18-05 ／ 検索

ニューラルモデルを使わずに、単語の重なりに基づいて類似文書を高速に検索したい。

In [1]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

data = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
docs = [d for d in data.data if len(d) > 200][:5000]
vec = TfidfVectorizer(stop_words="english", max_df=0.5, min_df=2, sublinear_tf=True)
X = vec.fit_transform(docs)   # 疎行列 (文書数 × 語彙数)。行は L2 正規化済み
print(X.shape, f"非ゼロ要素の割合 {X.nnz / (X.shape[0] * X.shape[1]):.4f}")

(5000, 27335) 非ゼロ要素の割合 0.0027


In [2]:
def search(query, k=3):
    sims = cosine_similarity(vec.transform([query]), X).ravel()
    for i in np.argsort(-sims)[:k]:
        print(f"  {sims[i]:.3f}  {' '.join(docs[i].split())[:90]}")

for q in ["space shuttle launch NASA", "best graphics card for windows"]:
    print(q)
    search(q)

space shuttle launch NASA
  0.383  For an essay, I am writing about the space shuttle and a need for a better propulsion syst
  0.344  This notice will be posted weekly in sci.space, sci.astro, and sci.space.shuttle. The Freq
  0.317  Hi all, I really thought that by now I would have seen something about this, but I haven't
best graphics card for windows
  0.339  I'm getting ready to buy a multimedia workstation and would like a little advice. I need a
  0.267  Preface: I am a novice user at best to the Windows environment. I am trying to execute a M
  0.241  I am ordering the Actix graphicsengine ultra plus. It is the same price as the stealth car


In [3]:
# 文書同士の類似度: 0 番の文書に似た文書
sims = cosine_similarity(X[0], X).ravel()
print(" ".join(docs[0].split())[:100])
for i in np.argsort(-sims)[1:3]:
    print(f"  {sims[i]:.3f}  {' '.join(docs[i].split())[:90]}")

I was wondering if anyone out there could enlighten me on this car I saw the other day. It was a 2-d
  0.229  Alright, beat this automobile sighting. Driving along just a hair north of Atlanta, I noti
  0.155  addition, Bricklins were manufactured in the 70s with engines from Ford. They are rather o


- 日本語の場合は `TfidfVectorizer(tokenizer=分かち書き関数, token_pattern=None)` とする。
- 文字 n-gram (`analyzer="char", ngram_range=(2, 3)`) にすると、分かち書きなしで表記ゆれにも強くなる。